# Spatial Routing Simulation V3 — Agent-Based Topology Sampling

**V3 changes over V2:**
- Topology states sampled via iterative agent simulation (levels 1/2/3) instead of uniform random
- `sampling_strategy` field saved per record (0=uniform, 1=sim_l1, 2=sim_l2, 3=sim_l3)
- Weights configurable via `STRAT_WEIGHTS` (default: 15% uniform, 25% sim_l1, 55% sim_l2, 5% sim_l3)
- Diagnostics cell breaks results down per sampling_strategy
- Link encoding: 9=active, 0=inactive, 8=node, 1/-1=cluster assignment


In [ ]:
import numpy as np, random, time, os, json, glob

try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName('SpatialRoutingSimV3').getOrCreate()
    print('PySpark session initialized. DataFrame API available.')
    try:
        sc = spark.sparkContext
    except Exception:
        print('SparkContext restricted (Unity Catalog Shared Cluster). Using DataFrame/Pandas UDF mode.')
        sc = None
except Exception:
    spark = None; sc = None
    print('Local mode')


In [ ]:
# Core Engine — graph topology solver with alpha-beta pruning and transposition cache

def build_topology_tables(rows, cols):
    h = 2*rows+1; w = 2*cols+1
    edge_to_bit = {}; bit_to_rc = {}; bit_to_label = {}; bit_idx = 0
    for r in range(h):
        for c in range(w):
            if r%2 == 0 and c%2 == 1:
                edge_to_bit[(r,c)] = bit_idx; bit_to_rc[bit_idx] = (r,c)
                bit_to_label[bit_idx] = f'H_{r}_{c}'; bit_idx += 1
            elif r%2 == 1 and c%2 == 0:
                edge_to_bit[(r,c)] = bit_idx; bit_to_rc[bit_idx] = (r,c)
                bit_to_label[bit_idx] = f'V_{r}_{c}'; bit_idx += 1
    n = bit_idx; all_mask = (1 << n) - 1
    box_masks = []
    for r in range(1, h, 2):
        for c in range(1, w, 2):
            t=edge_to_bit[(r-1,c)]; b=edge_to_bit[(r+1,c)]
            l=edge_to_bit[(r,c-1)]; rr=edge_to_bit[(r,c+1)]
            box_masks.append((1<<t)|(1<<b)|(1<<l)|(1<<rr))
    edge_boxes = [tuple(bm for bm in box_masks if bm & (1<<i)) for i in range(n)]
    labels = [bit_to_label[i] for i in range(n)]
    return n, all_mask, edge_boxes, labels, bit_to_rc, box_masks


def _closures(edges, i, edge_boxes):
    new = edges | (1<<i)
    return sum(1 for bm in edge_boxes[i] if (new & bm) == bm)


def _ordered_moves(edges, n_edges, edge_boxes):
    good = []; normal = []
    for i in range(n_edges):
        if edges & (1<<i): continue
        cl = _closures(edges, i, edge_boxes)
        (good if cl > 0 else normal).append((i, cl) if cl > 0 else i)
    return good, normal


def deep_evaluate(edges, depth, alpha, beta, maximizing, n_edges, all_mask, edge_boxes, tt):
    if depth == 0 or edges == all_mask: return 0
    key = (edges, depth, maximizing)
    cached = tt.get(key)
    if cached:
        f, v = cached
        if f == 0: return v
        if f == 1:
            if v >= beta: return v
            alpha = max(alpha, v)
        elif f == 2:
            if v <= alpha: return v
            beta = min(beta, v)
    good, normal = _ordered_moves(edges, n_edges, edge_boxes)
    orig_alpha = alpha
    if maximizing:
        best = -10000
        for move, cl in good:
            child = deep_evaluate(edges|(1<<move), depth-1, alpha-cl, beta-cl,
                                  True, n_edges, all_mask, edge_boxes, tt)
            best = max(best, cl+child); alpha = max(alpha, best)
            if beta <= alpha: break
        else:
            for move in normal:
                child = deep_evaluate(edges|(1<<move), depth-1, alpha, beta,
                                      False, n_edges, all_mask, edge_boxes, tt)
                best = max(best, child); alpha = max(alpha, best)
                if beta <= alpha: break
    else:
        best = 10000
        for move, cl in good:
            child = deep_evaluate(edges|(1<<move), depth-1, alpha+cl, beta+cl,
                                  False, n_edges, all_mask, edge_boxes, tt)
            best = min(best, -cl+child); beta = min(beta, best)
            if beta <= alpha: break
        else:
            for move in normal:
                child = deep_evaluate(edges|(1<<move), depth-1, alpha, beta,
                                      True, n_edges, all_mask, edge_boxes, tt)
                best = min(best, child); beta = min(beta, best)
                if beta <= alpha: break
    flag = 2 if best <= orig_alpha else (1 if best >= beta else 0)
    tt[key] = (flag, best)
    return best


def compute_all_scores(edges, depth, n_edges, all_mask, edge_boxes):
    tt = {}; scores = {}
    for i in range(n_edges):
        if edges & (1<<i): continue
        cl = _closures(edges, i, edge_boxes); new = edges | (1<<i)
        child = deep_evaluate(new, depth-1, -10001, 10001,
                              cl > 0, n_edges, all_mask, edge_boxes, tt)
        scores[i] = cl + child if cl > 0 else child
    return scores


def get_optimal_configuration(edges, depth, n_edges, all_mask, edge_boxes, labels):
    bit_scores = compute_all_scores(edges, depth, n_edges, all_mask, edge_boxes)
    label_scores = {labels[b]: v for b, v in bit_scores.items()}
    best_val = max(bit_scores.values())
    best_label = labels[random.choice([b for b, v in bit_scores.items() if v == best_val])]
    return best_label, label_scores


def edges_to_matrix(edges, rows, cols, n_edges, bit_to_rc, box_masks):
    h, w = 2*rows+1, 2*cols+1
    mat = np.zeros((h, w), dtype=np.int8)
    for r in range(0, h, 2):
        for c in range(0, w, 2): mat[r, c] = 8
    for i in range(n_edges):
        if edges & (1<<i): r, c = bit_to_rc[i]; mat[r, c] = 9
    for bm in box_masks:
        if (edges & bm) == bm:
            bits = [bit_to_rc[b] for b in range(n_edges) if bm & (1<<b)]
            ar = sum(rc[0] for rc in bits)//4
            ac = sum(rc[1] for rc in bits)//4
            if ar%2 == 1 and ac%2 == 1: mat[ar, ac] = 1
    return mat

print('Core engine loaded.')


In [ ]:
# V3: Agent-Based Topology Sampling
# sampling_strategy: 0=uniform, 1=sim_l1, 2=sim_l2, 3=sim_l3
# Adjust STRAT_WEIGHTS to change sampling distribution

STRAT_MODES  = [0,    1,    2,    3  ]  # strategy IDs
STRAT_WEIGHTS= [0.15, 0.25, 0.55, 0.05]  # [uniform, sim_l1, sim_l2, sim_l3]

MODE_NAMES  = {0: 'uniform', 1: 'sim_l1',
               2: 'sim_l2', 3: 'sim_l3'}


def _autoplay_edges(gen_depth, n_edges, all_mask, edge_boxes):
    """Two solvers at level gen_depth traverse the topology iteratively.
    Traversal stops at a random saturation between 15%-85%.
    Ties are broken via random selection among equivalent candidates.
    """
    target = random.randint(int(n_edges * 0.15), int(n_edges * 0.85))
    edges = 0; maximizing = True
    while bin(edges).count('1') < target and edges != all_mask:
        tt = {}; best_score = -99999 if maximizing else 99999
        best_moves = []
        for i in range(n_edges):
            if edges & (1<<i): continue
            cl = _closures(edges, i, edge_boxes)
            new = edges | (1<<i)
            child = deep_evaluate(
                new, gen_depth-1, -10001, 10001,
                (not maximizing) if cl == 0 else maximizing,
                n_edges, all_mask, edge_boxes, tt)
            score = cl + child if maximizing else -cl + child
            if score > best_score if maximizing else score < best_score:
                best_score = score; best_moves = [i]
            elif score == best_score:
                best_moves.append(i)  # keep ties for random selection
        if not best_moves: break
        best_move = random.choice(best_moves)  # random tie-breaking
        cl = _closures(edges, best_move, edge_boxes)
        edges |= (1 << best_move)
        if cl == 0: maximizing = not maximizing
    return edges


def generate_topology_v3(n_edges, all_mask, edge_boxes):
    """Returns (state_bitboard, sampling_strategy_int)."""
    mode = random.choices(STRAT_MODES, weights=STRAT_WEIGHTS)[0]
    if mode == 0:  # uniform random sampling
        qty = random.randint(int(n_edges * 0.15), int(n_edges * 0.85))
        idx = list(range(n_edges)); random.shuffle(idx)
        edges = 0
        for i in idx[:qty]: edges |= (1 << i)
    else:  # agent simulation at level=mode
        edges = _autoplay_edges(mode, n_edges, all_mask, edge_boxes)
    return edges, mode


print('V3 sampler loaded. Strategy weights:', dict(zip(MODE_NAMES.values(), STRAT_WEIGHTS)))


In [ ]:
# Execution Parameters

NUM_SAMPLES    = 300000
TAMANHO_LOTE   = 5000
DEPTH          = 8        # solver depth for scoring
ROWS, COLS     = 4, 3
SCORE_IND      = -1e9
DIRETORIO_SAIDA = '/Workspace/Users/c092820@corp.caixa.gov.br/CNN'
os.makedirs(DIRETORIO_SAIDA, exist_ok=True)

_n, _mask, _eboxes, _labels, _brc, _bmasks = build_topology_tables(ROWS, COLS)
_idx_label = {l: i for i, l in enumerate(_labels)}
_n_labels  = len(_labels)
print(f'Grid: {ROWS}x{COLS}, {_n} links | Output: {DIRETORIO_SAIDA}')
print(f'Sampling distribution: {dict(zip(MODE_NAMES.values(), STRAT_WEIGHTS))}')

# Expected generation_mode breakdown at 300k samples:
for m, w in zip(STRAT_MODES, STRAT_WEIGHTS):
    print(f'  {MODE_NAMES[m]}: ~{int(NUM_SAMPLES*w):,} records')


In [ ]:
# Distributed worker — factory pattern para evitar hardcoded depth/rows/cols.
# O worker é construído a partir dos parâmetros do driver (DEPTH, ROWS, COLS) que foram
# definidos no cell anterior, garantindo que todos os lotes usem exatamente a mesma
# profundidade declarada. Antes havia "depth=7" hardcoded aqui, e o dataset saía em
# profundidade diferente da anunciada (regressão silenciosa de 4h de Databricks).

def make_worker(depth, rows, cols):
    def process_batch_v3(iterator):
        import pandas as pd, numpy as np, random, json
        n, mask, eboxes, labels, brc, bms = build_topology_tables(rows, cols)
        for pdf in iterator:
            results = []
            for _ in range(len(pdf)):
                for _attempt in range(20):
                    try:
                        edges, mode = generate_topology_v3(n, mask, eboxes)
                        if edges == mask: continue
                        best, scores = get_optimal_configuration(
                            edges, depth, n, mask, eboxes, labels)
                        mat = edges_to_matrix(edges, rows, cols, n, brc, bms)
                        results.append({
                            'matriz': [int(x) for x in mat.flatten()],
                            'best_link': best,
                            'scores_dict': json.dumps(scores),
                            'generation_mode': int(mode),
                        })
                        break
                    except Exception:
                        pass
            yield pd.DataFrame(results)
    return process_batch_v3


def vetor_scores(sd, il, nl):
    v = np.full(nl, SCORE_IND, dtype=np.float32)
    for lbl, val in sd.items(): v[il[lbl]] = float(val)
    return v


def log_prog(gerados, total, inicio):
    dec = time.time() - inicio
    est = (dec / gerados * (total - gerados)) if gerados > 0 else 0
    print(json.dumps({'gerados': gerados, 'total': total,
                      'pct': round(gerados/total*100, 2),
                      'decorrido_s': round(dec, 2),
                      'restante_s': round(est, 2)}))

print(f'Worker factory pronto. Será instanciado com DEPTH={DEPTH}, ROWS={ROWS}, COLS={COLS}.')

In [ ]:
# Execution loop with checkpoint recovery
import pandas as pd

SCHEMA = 'matriz array<int>, best_link string, scores_dict string, generation_mode int'
CHUNK  = 2000
PARTS  = 128

arqs = sorted(glob.glob(os.path.join(DIRETORIO_SAIDA, 'dataset_pequeno_*.npz')))
if arqs:
    ul = int(arqs[-1].split('_')[-1].split('.')[0])
    total_gerado = ul * TAMANHO_LOTE
    print(f'Checkpoint: resuming from batch {ul} ({total_gerado} records)')
else:
    total_gerado = 0; ul = 0

estados = []; rotulos = []; scores_l = []; gen_modes = []
inicio = time.time()

print(f'Starting execution loop (Depth {DEPTH})...')
while total_gerado < NUM_SAMPLES:
    chunk = min(CHUNK, NUM_SAMPLES - total_gerado)
    if spark:
        df = spark.range(0, chunk, 1, min(PARTS, chunk))
        _worker = make_worker(DEPTH, ROWS, COLS)
        rows_out = df.mapInPandas(_worker, schema=SCHEMA).collect()
        for row in rows_out:
            mat = np.array(row.matriz, dtype=np.int8).reshape(2*ROWS+1, 2*COLS+1)
            sd  = json.loads(row.scores_dict)
            estados.append(mat)
            rotulos.append(row.best_link)
            scores_l.append(vetor_scores(sd, _idx_label, _n_labels))
            gen_modes.append(row.generation_mode)
            total_gerado += 1
    else:
        for _ in range(chunk):
            for _attempt in range(20):
                try:
                    edges, mode = generate_topology_v3(_n, _mask, _eboxes)
                    if edges == _mask: continue
                    best, sd = get_optimal_configuration(
                        edges, DEPTH, _n, _mask, _eboxes, _labels)
                    mat = edges_to_matrix(edges, ROWS, COLS, _n, _brc, _bmasks)
                    estados.append(mat); rotulos.append(best)
                    scores_l.append(vetor_scores(sd, _idx_label, _n_labels))
                    gen_modes.append(mode); total_gerado += 1; break
                except Exception: pass

    log_prog(total_gerado, NUM_SAMPLES, inicio)

    if len(estados) >= TAMANHO_LOTE or total_gerado >= NUM_SAMPLES:
        if estados:
            ul += 1
            path = os.path.join(DIRETORIO_SAIDA, f'dataset_pequeno_{ul:04d}.npz')
            np.savez_compressed(path,
                estados         = np.array(estados, dtype=np.int8),
                rotulos         = np.array(rotulos, dtype=str),
                scores          = np.array(scores_l, dtype=np.float32),
                generation_mode = np.array(gen_modes, dtype=np.int8),
                labels_canonicos= np.array(_labels, dtype=str),
                minimax_depth   = np.array([DEPTH], dtype=np.int32))
            print(f'Batch {ul:04d} -> {path}')
            estados = []; rotulos = []; scores_l = []; gen_modes = []

print(f'Completed: {total_gerado} records written to {DIRETORIO_SAIDA}')

In [ ]:
# Diagnostic Metrics — run after generation to inspect sampling distribution
import numpy as np, glob, os
try: import matplotlib.pyplot as plt; HAS_PLT = True
except ImportError: HAS_PLT = False

arqs = sorted(glob.glob(os.path.join(DIRETORIO_SAIDA, 'dataset_pequeno_*.npz')))
print(f'Files: {len(arqs)} | Estimated records: {len(arqs)*TAMANHO_LOTE:,}')

all_modes = []; all_fill = []; all_best_score = []; all_score_range = []
for f in arqs:
    d = np.load(f, allow_pickle=True)
    gm  = d['generation_mode']
    est = d['estados']
    sc  = d['scores']
    for i in range(len(gm)):
        mat = est[i]
        # count active links
        fill = 0
        h, w = mat.shape
        for r in range(h):
            for c in range(w):
                if (r%2==0 and c%2==1) or (r%2==1 and c%2==0):
                    if mat[r,c] != 0: fill += 1
        valid = sc[i][sc[i] > -1e8]
        bs = float(valid.max()) if len(valid) else 0.0
        sr = float(valid.max() - valid.min()) if len(valid) > 1 else 0.0
        all_modes.append(int(gm[i]))
        all_fill.append(fill)
        all_best_score.append(bs)
        all_score_range.append(sr)

all_modes = np.array(all_modes)
all_fill  = np.array(all_fill, dtype=float)
all_best_score  = np.array(all_best_score)
all_score_range = np.array(all_score_range)
total = len(all_modes)

print(f'Total records loaded: {total:,}')
print()
header = f'  {"Mode":<18}  {"N":>7}  {"% real":>7}  {"Fill_avg":>9}  {"BestScore":>10}  {"ScoreRange":>11}'
print(header); print('-' * len(header))
for m in sorted(set(all_modes.tolist())):
    mk = (all_modes == m)
    n  = int(mk.sum()); pct = n/total*100
    print(f'  {MODE_NAMES.get(m,str(m)):<18}  {n:>7}  {pct:>6.1f}%  '
          f'{all_fill[mk].mean():>9.1f}  {all_best_score[mk].mean():>10.2f}  '
          f'{all_score_range[mk].mean():>11.2f}')

if HAS_PLT:
    modes_present = sorted(set(all_modes.tolist()))
    colors = ['gray', 'steelblue', 'navy', 'purple']
    fig, axes = plt.subplots(len(modes_present), 2, figsize=(12, 4*len(modes_present)))
    if len(modes_present) == 1: axes = [axes]
    for idx, m in enumerate(modes_present):
        mk = (all_modes == m); c = colors[m % len(colors)]
        axes[idx][0].hist(all_fill[mk], bins=20, color=c, alpha=0.8)
        axes[idx][0].set_title(f'{MODE_NAMES.get(m,str(m))} — Link Saturation')
        axes[idx][0].set_xlabel('Active links'); axes[idx][0].set_ylabel('Freq')
        axes[idx][1].hist(all_best_score[mk], bins=20, color=c, alpha=0.8)
        axes[idx][1].set_title(f'{MODE_NAMES.get(m,str(m))} — Optimality Score')
        axes[idx][1].set_xlabel('Score'); axes[idx][1].set_ylabel('Freq')
    plt.suptitle('Sampling Strategy Diagnostics', fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print('matplotlib indisponivel: instale para ver graficos.')
